# Step 02 — Feature Engineering
**Defence Explanation:**  
Raw revenue data is just numbers in a table. Feature Engineering **transforms**  
that raw data into a form that a regression model can learn from.  
We create:  
- `time_index` — the numerical "X" axis (1, 2, 3… n). This is what Linear Regression needs.  
- `rolling_avg_3` — smooths out one-off spikes so the trend line is more stable.  
- `lag_1_revenue` — last period's revenue, a strong predictor of the next period's value.  

**Why this matters for defence:** You can say "I didn't just plug raw numbers into a formula —  
I engineered features that capture momentum (lag) and noise reduction (rolling average)."


In [2]:
import pandas as pd 
import numpy as np 

df = pd.read_csv("extracted_period_revenue.csv")
df["period_start_date"] = pd.to_datetime(df["period_start_date"])

print(f"✓ Loaded {len(df)} periods from extraction step")
df.head()


✓ Loaded 7 periods from extraction step


,period_id,period_name,period_start_date,actual_revenue,session_count,avg_shopee_rev,avg_tiktok_rev
0,1,Period 1,2025-03-03,393286038,86,3.907898e+06,665195.790698
1,2,Period 2,2025-04-05,622309567,147,3.964171e+06,269227.170068
2,3,Period 3,2025-05-02,653124767,200,2.968658e+06,296965.485000
3,4,Period 4,2025-08-30,531823266,171,2.799366e+06,310711.461988
4,5,Period 5,2025-10-01,419628196,180,2.250813e+06,80455.061111


In [4]:
# ── Feature 1: Time Index ─────────────────────────────────────────
# Linear Regression requires a numerical X axis.
# We assign each period an integer index: 1, 2, 3... n
# (Already sorted by period_id ascending from Step 01)

df["time_index"] = range(1, len(df) + 1)

# ── Feature 2: Rolling 3-Period Average ───────────────────────
# Smooths out single-period anomalies (e.g. a viral campaign spike).
# min_periods=1 ensures the first rows are not NaN.
df["rolling_avg_3"] = (
    df["actual_revenue"]
    .rolling(window=3, min_periods=1)
    .mean()
    .round(0)
)

# ── Feature 3: Lag-1 Revenue (previous period's revenue) ─────
# This is a powerful predictor: if last period was strong,
# this period is likely strong too (business momentum).
df["lag_1_revenue"] = df["actual_revenue"].shift(1)

# ── Feature 4: Period-over-Period Growth Rate ─────────────────
# Captures acceleration, not just level. A flat high is different
# from a rising low — both matter for forecasting direction.
df["revenue_growth_rate"] = df["actual_revenue"].pct_change().round(4)

# ── Drop the first row — it has NaN for lag features ─────────
df_features = df.dropna(subset=["lag_1_revenue"]).reset_index(drop=True)

print(f"✓ Feature engineering complete. {len(df_features)} usable rows (1 dropped for lag warmup)")
print(f"  Columns: {list(df_features.columns)}")
df_features.head()


✓ Feature engineering complete. 6 usable rows (1 dropped for lag warmup)
  Columns: ['period_id', 'period_name', 'period_start_date', 'actual_revenue', 'session_count', 'avg_shopee_rev', 'avg_tiktok_rev', 'time_index', 'rolling_avg_3', 'lag_1_revenue', 'revenue_growth_rate']


,period_id,period_name,period_start_date,actual_revenue,session_count,avg_shopee_rev,avg_tiktok_rev,time_index,rolling_avg_3,lag_1_revenue,revenue_growth_rate
0,2,Period 2,2025-04-05,622309567,147,3.964171e+06,269227.170068,2,507797802.0,393286038.0,0.5823
1,3,Period 3,2025-05-02,653124767,200,2.968658e+06,296965.485000,3,556240124.0,622309567.0,0.0495
2,4,Period 4,2025-08-30,531823266,171,2.799366e+06,310711.461988,4,602419200.0,653124767.0,-0.1857
3,5,Period 5,2025-10-01,419628196,180,2.250813e+06,80455.061111,5,534858743.0,531823266.0,-0.2110
4,6,Period 6,2025-10-31,170490846,110,1.481904e+06,68012.781818,6,373980769.0,419628196.0,-0.5937


In [5]:
# ── Save engineered features for Step 03 ─────────────────────────
df_features.to_csv("engineered_features.csv", index=False)

# Also save full df (including the first row) for export context
df.to_csv("all_periods_with_features.csv", index=False)

print("✓ Saved: engineered_features.csv")
print("✓ Saved: all_periods_with_features.csv")


✓ Saved: engineered_features.csv
✓ Saved: all_periods_with_features.csv
